<a href="https://colab.research.google.com/github/Mahendra2409/PyBlender/blob/main/Colab_Script/boy_01_pc_v2_colormaps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://drive.google.com/drive/folders/1sj-RqD5HRypGx-ZLqXpvyzY1CN84qJu-?usp=drive_link" target="_parent"><img src="https://img.shields.io/badge/PyBlender_Render_Farm-blue?logo=googledrive&logoColor=white" alt="PyBlender_Render_Farm"/></a>

In [ ]:
# Install wandb
!pip install wandb -q

# Authenticate securely using Kaggle Secrets
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api)

In [ ]:
# @title 1. Install Dependencies & Toolbox
# 1. Download & extract Blender (quiet)
!wget -nc -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
!tar -xf blender-4.0.2-linux-x64.tar.xz

# 2. Setup pip silently & Install dependencies (now includes google-cloud-storage)
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m ensurepip --upgrade > /dev/null 2>&1
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install \
    scipy matplotlib numpy wandb blendertoolbox plotly google-cloud-storage \
    -q --no-input --disable-pip-version-check

# 3. Download BlenderToolbox and move it to Kaggle's working directory
!git clone https://github.com/HTDerekLiu/BlenderToolbox.git
!mv BlenderToolbox/blendertoolbox /kaggle/working/


In [ ]:
# @title
# 3. Download BlenderToolbox
!git clone https://github.com/HTDerekLiu/BlenderToolbox.git
# Move the actual toolbox folder into your main /content/ directory so your script can 'see' it

!mv BlenderToolbox/blendertoolbox /content/
!./blender-4.0.2-linux-x64/4.0/python/bin/python3.10 -m pip install blendertoolbox

## Important Configuration Settings Checklist

These are the key settings that control the rendering process:

### Paths
- `DRIVE_BASE_PATH`: The base path in Google Drive where your PyBlender render farm data is stored. Example: `/content/drive/MyDrive/PyBlender_Render_Farm`
- `PC_TYPE`: The specific type or category of point cloud data being rendered. This defines a subfolder within `DRIVE_BASE_PATH`. Example: `ccylinder_scaled_PC_new`
- `GT_FILENAME`: The filename of the ground truth point cloud (`.xyz`) within the `PC_TYPE` folder. Example: `gt_ccylinder.xyz`
- `COLORMAP`: The Matplotlib colormap to use for rendering point cloud distances. Example: `viridis`
- `LOCAL_COLAB_BASE`: The local directory in Colab used for temporary storage of point cloud data copied from Drive. Example: `/content/local_data`

### Point Cloud Settings
- `PT_SIZE`: The size of each point in the rendered point cloud, often interpreted as a radius if the points are rendered as spheres. Example: `0.009` (for a radius of 0.009 units)
- `PT_COLOR`: A RGBA tuple defining the default color of the points, if not overridden by colormaps. Example: `[0.5, 1.0, 1.0, 0.0, 0.0]` (note: BlenderToolbox often uses a specific format, ensure it matches expectations).

### Object Transforms
- `OBJ_LOCATION`: A 3-tuple representing the X, Y, Z coordinates for the object's position in Blender space. Example: `(-0.370186, -5.31554, -2.55748)`
- `OBJ_ROTATION`: A 3-tuple representing the X, Y, Z rotation angles (in degrees) of the object. Example: `(448.706, -0.984887, -94.562)`
- `OBJ_SCALE`: A 3-tuple representing the X, Y, Z scaling factors applied to the object. Example: `(3.71272, 3.71272, 3.71272)`

## Interactive Configuration Form

Use the form fields below to easily adjust the key rendering parameters. Once you've made your changes, run this cell to update the `CONFIG` dictionary in `config.py`.

In [ ]:
#@title 2. Authenticate Google Cloud Storage
import os
import json
from kaggle_secrets import UserSecretsClient

# 1. Retrieve the GCS service account key from Kaggle Secrets
user_secrets = UserSecretsClient()
gcs_key_json = user_secrets.get_secret("GCS_SERVICE_ACCOUNT_KEY")

# 2. Write it to a file that Blender's Python can read
GCS_KEY_PATH = "/tmp/gcs_service_account.json"
with open(GCS_KEY_PATH, "w") as f:
    f.write(gcs_key_json)

# 3. Set environment variable (Blender inherits this from the parent process)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCS_KEY_PATH

print(f"GCS credentials saved to {GCS_KEY_PATH}")
print(f"GOOGLE_APPLICATION_CREDENTIALS = {os.environ['GOOGLE_APPLICATION_CREDENTIALS']}")

# 4. Quick verification
from google.cloud import storage
client = storage.Client()
bucket_name = "pyblender-render-farm"  # <-- match your config
bucket = client.bucket(bucket_name)
if bucket.exists():
    print(f"SUCCESS: Connected to gs://{bucket_name}")
else:
    print(f"WARNING: Bucket '{bucket_name}' not found. Check the name.")


In [ ]:
%%writefile config.py

# ==========================================
# 1. MAIN CONFIGURATION
# ==========================================

CONFIG = {
    # --- Output Controls ---
    "SAVE_BLEND_FILE": False,   
    "FORCE_OVERWRITE": True,   

    # --- Kaggle Paths ---
    "KAGGLE_INPUT_DIR": "/kaggle/input/datasets/mahendra2409/pyblender-pointclouds/xyzFormat",
    "KAGGLE_OUTPUT_DIR": "/kaggle/working/RenderImages",

    "PC_TYPE": "boy_01_PC_v2",
    "GT_FILENAME": "boy01.xyz",

    # --- Google Cloud Storage ---
    "GCS_BUCKET_NAME": "pyblender-render-farm",  # <-- CHANGE THIS to your actual bucket name
    "GCS_BASE_PATH": "RenderImages",             # Base folder path inside the bucket

    # --- Colormaps ---
    "COLORMAPS": ['Accent', 'Accent_r', 'Blues', 'Blues_r', 'BrBG', 'BrBG_r', 'BuGn', 'BuGn_r', 'BuPu', 'BuPu_r', 'CMRmap', 'CMRmap_r', 'Dark2', 'Dark2_r', 'GnBu', 'GnBu_r', 'Grays', 'Grays_r', 'Greens', 'Greens_r', 'Greys', 'Greys_r', 'OrRd', 'OrRd_r', 'Oranges', 'Oranges_r', 'PRGn', 'PRGn_r', 'Paired', 'Paired_r', 'Pastel1', 'Pastel1_r', 'Pastel2', 'Pastel2_r', 'PiYG', 'PiYG_r', 'PuBu', 'PuBuGn', 'PuBuGn_r', 'PuBu_r', 'PuOr', 'PuOr_r', 'PuRd', 'PuRd_r', 'Purples', 'Purples_r', 'RdBu', 'RdBu_r', 'RdGy', 'RdGy_r', 'RdPu', 'RdPu_r', 'RdYlBu', 'RdYlBu_r', 'RdYlGn', 'RdYlGn_r', 'Reds', 'Reds_r', 'Set1', 'Set1_r', 'Set2', 'Set2_r', 'Set3', 'Set3_r', 'Spectral', 'Spectral_r', 'Wistia', 'Wistia_r', 'YlGn', 'YlGnBu', 'YlGnBu_r', 'YlGn_r', 'YlOrBr', 'YlOrBr_r', 'YlOrRd', 'YlOrRd_r', 'afmhot', 'afmhot_r', 'autumn', 'autumn_r', 'berlin', 'berlin_r', 'binary', 'binary_r', 'bone', 'bone_r', 'brg', 'brg_r', 'bwr', 'bwr_r', 'cividis', 'cividis_r', 'cool', 'cool_r', 'coolwarm', 'coolwarm_r', 'copper', 'copper_r', 'cubehelix', 'cubehelix_r', 'flag', 'flag_r', 'gist_earth', 'gist_earth_r', 'gist_gray', 'gist_gray_r', 'gist_grey', 'gist_grey_r', 'gist_heat', 'gist_heat_r', 'gist_ncar', 'gist_ncar_r', 'gist_rainbow', 'gist_rainbow_r', 'gist_stern', 'gist_stern_r', 'gist_yarg', 'gist_yarg_r', 'gist_yerg', 'gist_yerg_r', 'gnuplot', 'gnuplot2', 'gnuplot2_r', 'gnuplot_r', 'gray', 'gray_r', 'grey', 'grey_r', 'hot', 'hot_r', 'hsv', 'hsv_r', 'inferno', 'inferno_r', 'jet', 'jet_r', 'magma', 'magma_r', 'managua', 'managua_r', 'nipy_spectral', 'nipy_spectral_r', 'ocean', 'ocean_r', 'pink', 'pink_r', 'plasma', 'plasma_r', 'prism', 'prism_r', 'rainbow', 'rainbow_r', 'seismic', 'seismic_r', 'spring', 'spring_r', 'summer', 'summer_r', 'tab10', 'tab10_r', 'tab20', 'tab20_r', 'tab20b', 'tab20b_r', 'tab20c', 'tab20c_r', 'terrain', 'terrain_r', 'turbo', 'turbo_r', 'twilight', 'twilight_r', 'twilight_shifted', 'twilight_shifted_r', 'vanimo', 'vanimo_r', 'viridis', 'viridis_r', 'winter', 'winter_r'],

    # --- Point Cloud Settings ---
    "PT_SIZE": 0.81,
    "PT_COLOR": [0.5, 1.0, 1.0, 0.0, 0.0],

    # --- Object Transforms ---
    "OBJ_LOCATION": (7.28883, -6.13761, -2.16309),
    "OBJ_ROTATION": (92.8616, -6.26444, 62.9924),
    "OBJ_SCALE": (0.013583, 0.013583, 0.013583),

    # --- Blender Render Settings ---
    "IMG_RES_X": 1000,
    "IMG_RES_Y": 1000,
    "NUM_SAMPLES": 100,
    "EXPOSURE": 1.5,

    # --- Camera Settings ---
    "CAM_LOCATION": (3, 0, 2),
    "LOOK_AT": (0, 0, 0.5),
    "FOCAL_LENGTH": 45,

    # --- Light Settings ---
    "LIGHT_ANGLE": (6, -30, -155),
    "LIGHT_STRENGTH": 2,
    "SHADOW_SOFTNESS": 0.3,
    "AMBIENT_COLOR": (0.1, 0.1, 0.1, 1),
    "SHADOW_THRESHOLD": 0.05
}

GT_OVERRIDES = {}


In [ ]:
%%writefile render.py

import os
import sys
import time
import threading
import queue
import numpy as np
from scipy.spatial import cKDTree

if '/kaggle/working' not in sys.path:
    sys.path.append('/kaggle/working')
from config import CONFIG, GT_OVERRIDES

os.environ['MPLBACKEND'] = 'Agg'
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import bpy
import blendertoolbox as bt
import wandb


# ==========================================
# ASYNC GCS UPLOADER (background thread)
# ==========================================
class AsyncGCSUploader:
    """Uploads files to GCS in a background thread so rendering isn't blocked."""
    def __init__(self, bucket):
        self.bucket = bucket
        self.queue = queue.Queue()
        self.uploads = 0
        self.failures = 0
        self._stop = False

        if bucket is not None:
            self.thread = threading.Thread(target=self._worker, daemon=True)
            self.thread.start()
        else:
            self.thread = None

    def _worker(self):
        while not self._stop or not self.queue.empty():
            try:
                local_path, gcs_path = self.queue.get(timeout=1)
                try:
                    blob = self.bucket.blob(gcs_path)
                    blob.upload_from_filename(local_path)
                    self.uploads += 1
                    print(f"  >> Uploaded to gs://{CONFIG['GCS_BUCKET_NAME']}/{gcs_path}")
                except Exception as e:
                    self.failures += 1
                    print(f"  >> GCS upload failed: {e}")
                self.queue.task_done()
            except queue.Empty:
                continue

    def upload(self, local_path, gcs_path):
        if self.bucket is not None:
            self.queue.put((local_path, gcs_path))

    def wait_and_stop(self):
        if self.thread is not None:
            self.queue.join()  # Wait for all uploads to finish
            self._stop = True
            self.thread.join(timeout=10)


# ==========================================
# GOOGLE CLOUD STORAGE SETUP
# ==========================================
def setup_gcs():
    try:
        from google.cloud import storage
        key_path = CONFIG.get("GCS_KEY_PATH", "/tmp/gcs_service_account.json")
        if not os.path.exists(key_path):
            print(f"  GCS key file not found at {key_path}")
            return None
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = key_path
        client = storage.Client()
        bucket = client.bucket(CONFIG["GCS_BUCKET_NAME"])
        try:
            next(bucket.list_blobs(max_results=1), None)
            print(f"  GCS connected! Bucket: gs://{CONFIG['GCS_BUCKET_NAME']}")
        except Exception:
            print(f"  WARNING: Could not verify bucket, uploads may still work.")
        return bucket
    except Exception as e:
        print(f"  ERROR setting up GCS: {e}")
        print(f"  Renders will be saved locally only.")
        return None


# ==========================================
# COLORMAP VALIDATION
# ==========================================
def validate_colormaps(colormap_list):
    valid_colormaps = []
    invalid_colormaps = []
    for cmap_name in colormap_list:
        try:
            plt.get_cmap(cmap_name)
            valid_colormaps.append(cmap_name)
        except (ValueError, KeyError):
            invalid_colormaps.append(cmap_name)
    if invalid_colormaps:
        print(f"\n{'='*50}")
        print(f"  WARNING: {len(invalid_colormaps)} invalid colormap(s) will be SKIPPED")
        print(f"{'='*50}")
        for name in invalid_colormaps:
            print(f"    X  '{name}'")
        print(f"  Proceeding with {len(valid_colormaps)} valid colormap(s).\n")
    if not valid_colormaps:
        print(f"  ERROR: No valid colormaps found!")
    return valid_colormaps


# ==========================================
# GPU OPTIMIZATION
# ==========================================
def force_multi_gpu():
    scene = bpy.context.scene
    scene.render.engine = 'CYCLES'
    scene.cycles.device = 'GPU'
    prefs = bpy.context.preferences.addons['cycles'].preferences

    try:
        prefs.compute_device_type = 'OPTIX'
    except TypeError:
        prefs.compute_device_type = 'CUDA'

    prefs.get_devices()
    for device in prefs.devices:
        if device.type in ['OPTIX', 'CUDA']:
            device.use = True
        else:
            device.use = False

    # OptiX Denoiser (GPU)
    scene.cycles.use_denoising = True
    try:
        scene.cycles.denoiser = 'OPTIX'
    except Exception:
        try:
            scene.cycles.denoiser = 'OPENIMAGEDENOISE'
        except Exception:
            pass

    # Persistent Data
    scene.render.use_persistent_data = True

    # GPU Compositing
    try:
        scene.render.compositor_device = 'GPU'
    except Exception:
        pass

    # Tile size for GPU
    try:
        scene.cycles.tile_size = 256
    except Exception:
        pass

    # Fast GI
    try:
        scene.cycles.use_fast_gi = True
    except Exception:
        pass


# ==========================================
# DERIVED PATHS
# ==========================================
SOURCE_DIR = os.path.join(CONFIG["KAGGLE_INPUT_DIR"], CONFIG["PC_TYPE"])
GROUND_TRUTH_PATH = os.path.join(SOURCE_DIR, CONFIG["GT_FILENAME"])


def render_point_clouds():
    print("--- Starting Render Process ---")

    if not os.path.exists(SOURCE_DIR):
        print(f"ERROR: Cannot find input directory: {SOURCE_DIR}")
        return

    valid_colormaps = validate_colormaps(CONFIG["COLORMAPS"])
    if not valid_colormaps:
        return

    print("\n--- Setting up Google Cloud Storage ---")
    gcs_bucket = setup_gcs()

    # Start async uploader
    uploader = AsyncGCSUploader(gcs_bucket)

    wandb.init(
        project="pyblender-render-farm",
        name=f"Render_{CONFIG['PC_TYPE']}",
        config=CONFIG
    )

    # Pre-load ground truth
    print("\n--- Pre-loading ground truth ---")
    ground_truth_points = np.loadtxt(GROUND_TRUTH_PATH)[:, :3]
    tree = cKDTree(ground_truth_points)
    print(f"  Ground truth: {ground_truth_points.shape[0]} points loaded")

    # Pre-load ALL point clouds and pre-compute distances ONCE
    print("\n--- Pre-loading all point clouds & computing distances ---")
    all_files = [f for f in os.listdir(SOURCE_DIR) if f.endswith(".xyz")]
    file_data = {}

    for filename in all_files:
        is_gt = (filename == CONFIG["GT_FILENAME"])
        if is_gt:
            file_data[filename] = {
                "points": ground_truth_points,
                "is_gt": True,
                "normalized_distances": None
            }
        else:
            points = np.loadtxt(os.path.join(SOURCE_DIR, filename))[:, :3]
            distances, _ = tree.query(points)
            max_distance = np.percentile(distances, 99)
            normalized_distances = np.clip(distances / max_distance, 0, 1)
            normalized_distances = np.log1p(normalized_distances) / np.log1p(1)
            file_data[filename] = {
                "points": points,
                "is_gt": False,
                "normalized_distances": normalized_distances
            }
        num_pts = file_data[filename]["points"].shape[0]
        label = "GT" if is_gt else f"{num_pts} pts"
        print(f"  Loaded: {filename} ({label})")

    print(f"  All {len(all_files)} files pre-loaded!\n")

    # Progress tracking
    total_renders = len(all_files) * len(valid_colormaps)
    completed_renders = 0

    # >>> OPTIMIZED LOOP: file-first, colormap-second <<<
    # Each file is loaded ONCE, distances computed ONCE
    # Only the colormap application changes per iteration
    for filename in all_files:
        data = file_data[filename]
        is_gt = data["is_gt"]

        for current_colormap in valid_colormaps:
            active_cfg = CONFIG.copy()
            if is_gt:
                active_cfg.update(GT_OVERRIDES)
            active_cfg["COLORMAP"] = current_colormap

            output_filename = f"{os.path.splitext(filename)[0]}.png"
            local_output_dir = os.path.join(CONFIG["KAGGLE_OUTPUT_DIR"], CONFIG["PC_TYPE"], f"{current_colormap}_colormap")
            os.makedirs(local_output_dir, exist_ok=True)
            render_output = os.path.join(local_output_dir, output_filename)

            gcs_colormap_dir = f"{CONFIG['GCS_BASE_PATH']}/{CONFIG['PC_TYPE']}/{current_colormap}_colormap"
            gcs_blob_path = f"{gcs_colormap_dir}/{output_filename}"

            # Check GCS
            if gcs_bucket and not active_cfg["FORCE_OVERWRITE"]:
                blob = gcs_bucket.blob(gcs_blob_path)
                if blob.exists():
                    print(f"Already on GCS: {output_filename} ({current_colormap}). Skipping...")
                    completed_renders += 1
                    continue

            # Check local
            if os.path.exists(render_output) and not active_cfg["FORCE_OVERWRITE"]:
                print(f"Skipping {filename} ({current_colormap})...")
                completed_renders += 1
                continue

            start_time = time.time()

            # Blender Setup
            bt.blenderInit(
                active_cfg["IMG_RES_X"], active_cfg["IMG_RES_Y"],
                active_cfg["NUM_SAMPLES"], active_cfg["EXPOSURE"]
            )
            force_multi_gpu()

            # Apply colormap to PRE-COMPUTED data (super fast — no file I/O or tree.query)
            try:
                colormap = plt.get_cmap(active_cfg["COLORMAP"])
                if is_gt:
                    min_color = colormap(0.0)[:3]
                    colors = np.tile(min_color, (data["points"].shape[0], 1))
                else:
                    colors = colormap(data["normalized_distances"])[:, :3]
            except (ValueError, KeyError) as e:
                print(f"  ERROR: Invalid colormap '{active_cfg['COLORMAP']}' — {e}. Skipping.")
                continue

            mesh = bt.readNumpyPoints(data["points"], active_cfg["OBJ_LOCATION"], active_cfg["OBJ_ROTATION"], active_cfg["OBJ_SCALE"])
            mesh = bt.setPointColors(mesh, colors)

            ptColor = bt.colorObj(active_cfg["PT_COLOR"], 0.5, 1.0, 1.0, 0.0, 0.0)
            bt.setMat_pointCloudColored(mesh, ptColor, active_cfg["PT_SIZE"])

            cam = bt.setCamera(active_cfg["CAM_LOCATION"], active_cfg["LOOK_AT"], active_cfg["FOCAL_LENGTH"])
            sun = bt.setLight_sun(active_cfg["LIGHT_ANGLE"], active_cfg["LIGHT_STRENGTH"], active_cfg["SHADOW_SOFTNESS"])
            bt.setLight_ambient(color=active_cfg["AMBIENT_COLOR"])
            bt.shadowThreshold(alphaThreshold=active_cfg["SHADOW_THRESHOLD"], interpolationMode='CARDINAL')

            # Simplified compositor (denoising runs in Cycles via OptiX)
            bpy.context.scene.use_nodes = True
            tree_nodes = bpy.context.scene.node_tree
            tree_nodes.nodes.clear()

            render_layers = tree_nodes.nodes.new('CompositorNodeRLayers')
            composite = tree_nodes.nodes.new('CompositorNodeComposite')
            viewer = tree_nodes.nodes.new('CompositorNodeViewer')

            render_layers.location = (-300, 0)
            composite.location = (300, 0)
            viewer.location = (300, -200)

            tree_nodes.links.new(render_layers.outputs['Image'], composite.inputs['Image'])
            tree_nodes.links.new(render_layers.outputs['Image'], viewer.inputs['Image'])

            # Render
            bt.renderImage(render_output, cam)

            # Async upload — doesn't block next render
            uploader.upload(render_output, gcs_blob_path)

            # Log to WandB
            render_duration = time.time() - start_time
            completed_renders += 1
            progress_pct = (completed_renders / total_renders) * 100

            wandb.log({
                "progress_percent": progress_pct,
                "render_time_seconds": render_duration,
                "current_colormap": current_colormap,
                "filename": filename,
                "gcs_uploads_total": uploader.uploads,
                "latest_render": wandb.Image(render_output)
            })
            print(f"Render: {filename} ({current_colormap}) in {render_duration:.1f}s  [{completed_renders}/{total_renders} = {progress_pct:.0f}%]")

    # Wait for remaining uploads
    print("\n--- Waiting for remaining GCS uploads ---")
    uploader.wait_and_stop()

    print(f"\n{'='*50}")
    print(f"  RENDER COMPLETE!")
    print(f"  Total renders: {completed_renders}/{total_renders}")
    print(f"  GCS uploads: {uploader.uploads} (failures: {uploader.failures})")
    print(f"  Bucket: gs://{CONFIG['GCS_BUCKET_NAME']}/{CONFIG['GCS_BASE_PATH']}/{CONFIG['PC_TYPE']}/")
    print(f"{'='*50}\n")

    wandb.finish()

if __name__ == "__main__":
    render_point_clouds()


In [ ]:
#@title 4. Start Rendering Render

!./blender-4.0.2-linux-x64/blender -b -P render.py

# Once finished, zip the output directory so you can download it from Kaggle
!zip -r RenderImages.zip /kaggle/working/RenderImages

In [ ]:
#@title Upload existing local renders to GCS
import os
from google.cloud import storage

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/tmp/gcs_service_account.json"
client = storage.Client()
bucket = client.bucket("pyblender-render-farm")  # <-- your bucket name

local_base = "/kaggle/working/RenderImages"
gcs_base = "RenderImages"
uploaded = 0

for root, dirs, files in os.walk(local_base):
    for f in files:
        if f.endswith(".png"):
            local_path = os.path.join(root, f)
            relative = os.path.relpath(local_path, local_base)
            gcs_path = f"{gcs_base}/{relative}"
            
            blob = bucket.blob(gcs_path)
            if not blob.exists():
                blob.upload_from_filename(local_path)
                uploaded += 1
                print(f"  Uploaded: {gcs_path}")

print(f"\nDone! Uploaded {uploaded} files to gs://pyblender-render-farm/{gcs_base}/")


In [ ]:
!ls -R /kaggle/input/